# 🏏 IPL Data Analysis — Exploratory Data Analysis

Complete EDA on IPL 2008–2023 dataset covering match statistics, player performance, and team insights.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')

# Style
plt.style.use('dark_background')
sns.set_palette('husl')
plt.rcParams.update({'figure.figsize':(12,5), 'axes.titlesize':14, 'font.family':'sans-serif'})
print('Libraries loaded.')

## 1. Load & Clean Data

In [ ]:
from utils.data_loader import load_matches, load_deliveries, merge_data
from utils.data_loader import compute_batsman_stats, compute_bowler_stats, compute_team_stats

matches    = load_matches()
deliveries = load_deliveries()
merged     = merge_data(matches, deliveries)

print(f'Matches:     {matches.shape}')
print(f'Deliveries:  {deliveries.shape}')
print(f'Merged:      {merged.shape}')
matches.head()

In [ ]:
# Data quality check
print('=== Matches null summary ===')
print(matches.isnull().sum())
print('\n=== Deliveries null summary ===')
print(deliveries.isnull().sum())
print('\n=== Matches dtypes ===')
print(matches.dtypes)

## 2. Basic Match Analysis

In [ ]:
# Matches per season
mps = matches.groupby('season').size().reset_index(name='matches')
fig, ax = plt.subplots()
ax.bar(mps['season'], mps['matches'], color='#f97316', alpha=0.85)
ax.set_title('Matches Per Season')
ax.set_xlabel('Season'); ax.set_ylabel('Matches')
plt.tight_layout(); plt.show()

In [ ]:
# Toss decision distribution
td = matches['toss_decision'].value_counts()
fig, axes = plt.subplots(1, 2, figsize=(12,4))

axes[0].pie(td.values, labels=td.index, autopct='%1.1f%%',
            colors=['#f97316','#3b82f6'], startangle=140)
axes[0].set_title('Toss Decision')

# Toss win vs match win
tw_mw = matches[matches['toss_winner']==matches['winner']]
pct = round(len(tw_mw)/len(matches)*100, 1)
axes[1].bar(['Toss=Match Win','Toss≠Match Win'], [pct, 100-pct],
            color=['#22c55e','#ef4444'])
axes[1].set_title('Toss Winner = Match Winner %')
axes[1].set_ylabel('%')
plt.tight_layout(); plt.show()
print(f'Toss-to-Win conversion: {pct}%')

## 3. Team Analysis

In [ ]:
team_stats = compute_team_stats(matches)
print(team_stats[['team','matches_played','wins','losses','win_pct']].to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(12,5))
ts = team_stats.sort_values('win_pct', ascending=True)
bars = ax.barh(ts['team'], ts['win_pct'], color=plt.cm.RdYlGn(
    [v/100 for v in ts['win_pct']]))
ax.set_title('Win Percentage by Team (All Time)')
ax.set_xlabel('Win %')
for bar, val in zip(bars, ts['win_pct']):
    ax.text(bar.get_width()+0.5, bar.get_y()+bar.get_height()/2,
            f'{val}%', va='center', fontsize=9)
plt.tight_layout(); plt.show()

## 4. Batsman Analysis

In [ ]:
bat_stats = compute_batsman_stats(deliveries)
top10_bat = bat_stats.nlargest(10, 'total_runs')
print(top10_bat[['player','total_runs','balls_faced','strike_rate','average','fours','sixes']].to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14,5))

# Top 10 by runs
axes[0].barh(top10_bat['player'][::-1], top10_bat['total_runs'][::-1],
             color='#f97316')
axes[0].set_title('Top 10 Batsmen — Total Runs')
axes[0].set_xlabel('Runs')

# Strike rate scatter
axes[1].scatter(top10_bat['total_runs'], top10_bat['strike_rate'],
                s=120, c='#3b82f6', alpha=0.8)
for _, row in top10_bat.iterrows():
    axes[1].annotate(row['player'].split()[-1],
                     (row['total_runs'], row['strike_rate']),
                     fontsize=8, textcoords='offset points', xytext=(5,5))
axes[1].set_title('Runs vs Strike Rate')
axes[1].set_xlabel('Total Runs'); axes[1].set_ylabel('Strike Rate')
plt.tight_layout(); plt.show()

## 5. Bowler Analysis

In [ ]:
bowl_stats = compute_bowler_stats(deliveries)
top10_bowl = bowl_stats[bowl_stats['wickets']>0].nlargest(10,'wickets')
print(top10_bowl[['player','wickets','overs_bowled','economy_rate','bowling_average']].to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14,5))

axes[0].barh(top10_bowl['player'][::-1], top10_bowl['wickets'][::-1],
             color='#22c55e')
axes[0].set_title('Top 10 Bowlers — Wickets')
axes[0].set_xlabel('Wickets')

axes[1].scatter(top10_bowl['wickets'], top10_bowl['economy_rate'],
                s=120, c='#a855f7', alpha=0.8)
for _, row in top10_bowl.iterrows():
    axes[1].annotate(row['player'].split()[-1],
                     (row['wickets'], row['economy_rate']),
                     fontsize=8, textcoords='offset points', xytext=(5,5))
axes[1].set_title('Wickets vs Economy Rate')
axes[1].set_xlabel('Wickets'); axes[1].set_ylabel('Economy')
plt.tight_layout(); plt.show()

## 6. Venue & Tournament Insights

In [ ]:
# Top venues
vc = matches['venue'].value_counts().head(10)
fig, ax = plt.subplots(figsize=(12,4))
ax.barh(vc.index[::-1], vc.values[::-1], color='#6366f1')
ax.set_title('Top 10 Venues by Matches Hosted')
ax.set_xlabel('Matches')
plt.tight_layout(); plt.show()

In [ ]:
# Batting first vs chasing
result_counts = matches['result'].value_counts()
print('Result breakdown:')
print(result_counts)

bat_first_win = len(matches[matches['result']=='runs'])
chase_win     = len(matches[matches['result']=='wickets'])
total         = bat_first_win + chase_win
print(f'\nBatting First Win%: {bat_first_win/total*100:.1f}%')
print(f'Chasing Win%:       {chase_win/total*100:.1f}%')

## 7. Correlation Heatmap

In [ ]:
numeric_cols = bat_stats[['total_runs','balls_faced','strike_rate','average','fours','sixes','matches']]
fig, ax = plt.subplots(figsize=(9,7))
sns.heatmap(numeric_cols.corr(), annot=True, fmt='.2f',
            cmap='coolwarm', ax=ax, linewidths=0.5)
ax.set_title('Batsman Stats Correlation')
plt.tight_layout(); plt.show()

## 8. Win Prediction Model

In [ ]:
from models.predictor import build_model, predict_winner

model, encoders, accuracy, report = build_model(matches)
print(f'Model Accuracy: {accuracy*100:.2f}%')
print('\nClassification Report:')
print(report)

In [ ]:
winner, conf = predict_winner(
    model, encoders,
    team1='Mumbai Indians',
    team2='Chennai Super Kings',
    toss_winner='Mumbai Indians',
    toss_decision='bat',
    venue='Wankhede Stadium, Mumbai'
)
print(f'Predicted Winner: {winner}  (Confidence: {conf}%)')